In [13]:
# 导入工具
from openai import OpenAI
from dotenv import load_dotenv
import os

In [14]:
# 读取 API key
load_dotenv()

# 创建客户端（通义千问）
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("API_KEY")
)

In [15]:
# 一个最简单的函数：问 AI 一个问题
def ask_ai(prompt):
    response = client.chat.completions.create(
        model="qwen3.6-plus",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [16]:
# 定义需求
需求 = "希望系统可以自动预测产品从研发到产线的一次合格率"

In [17]:
#第一步，先让AI理解这个需求“本质在干嘛”
step1 = ask_ai(f"""
需求是：{需求}

请用简单中文分析：
这个需求本质是在做什么？
是规则处理、数据展示，还是预测/判断？
""")

print(step1)

这个需求本质是：**利用历史数据和产品/工艺参数，提前推算出产品量产时的质量表现（一次合格率）**。

它明确属于 **预测/判断**（更准确说是“预测”）。

简单对比说明：
- ❌ 不是**规则处理**：合格率受材料、工艺、设备、人员、环境等大量因素交叉影响，关系复杂且非线性，无法用固定的“如果A就B”逻辑直接套算。
- ❌ 不是**数据展示**：系统不是把已经发生的质量数据画成报表给你看，而是在还没量产（或刚进产线）时，提前“算”出一个未来可能的结果。
- ✅ 是**预测/判断**：需要系统学习历史研发数据、试产记录、工艺参数等，通过统计或算法模型，输出一个“大概率会达到的合格率数值或区间”。

一句话总结：这是用历史数据+算法做**前瞻性测算**，属于典型的预测类需求。


In [18]:
#第二步，判断这个需求适合做成 AI项目 还是 IT项目

step2 = ask_ai(f"""
这是对需求的理解：

{step1}

请判断这个需求更适合做成 AI项目 还是 IT项目。

请按下面格式回答：
1. 判断结果
2. 原因
3. 为什么不是另一种
""")

print(step2)

1. 判断结果
更适合做成 **AI项目**。

2. 原因
- **核心能力匹配**：该需求本质是“从历史数据中学习规律并输出未来概率”，属于典型的监督学习/回归预测场景，与机器学习（如树模型、回归算法、深度学习）的能力边界高度一致。
- **擅长处理非线性与高维交互**：一次合格率受材料批次、设备状态、工艺窗口、人员操作、环境波动等多变量耦合影响，AI模型可通过特征交叉、自动特征选择与非线性激活，挖掘人脑或传统公式难以刻画的隐藏规律。
- **输出形态天然契合**：AI模型默认输出带置信区间或概率分布的结果（如“预估一次合格率 92.3% ±1.5%，置信度 90%”），符合制造工程中对风险量化和决策容错的实际需求。
- **具备持续进化特性**：量产数据持续产生后，AI可通过在线学习或定期重训实现模型迭代，预测精度随数据积累而提升，形成“数据飞轮”。

3. 为什么不是另一种（IT项目）
- **IT项目的底层逻辑是“确定性规则与流程”**：传统IT系统以数据库+固定业务逻辑（CRUD、审批流、阈值判断、报表引擎）为核心，只能执行预设的SQL或硬编码公式，无法从数据中“自学”未知关联，更无法对未发生过的新工艺组合进行合理外推。
- **无法满足“预测”诉求**：IT系统擅长“事后记录”与“事中监控”，但本需求明确要求“事前推算”。若强行用IT实现，只能退化为静态Excel公式库或经验规则表，一旦遇到复杂工况或参数超出历史范围，输出结果将严重失真且无法量化误差。
- **交付范式与评估标准不同**：IT项目以功能是否上线、流程是否跑通、响应是否稳定为验收标准；而本需求需经历数据治理、特征工程、模型训练、交叉验证、线上监控等数据科学链路，核心指标是预测准确率、泛化能力与业务ROI，属于标准的AI项目生命周期。


In [19]:
# 第三步，如果要落地，这个需求应该怎么做？需要什么数据？最小版本是什么样的？应该先找谁参与？
step3 = ask_ai(f"""
需求：{需求}

需求理解：{step1}

判断结果：{step2}

如果这个需求要真正落地，请给出一个简单建议：
1. 需要什么数据
2. 适合先做什么最小版本
3. 应该先找谁参与

用简单中文回答
""")

print(step3)

以下是针对该需求落地的3条简明建议：

**1. 需要什么数据**
核心是能按**“产品/工单/批次”串联起来**的三类数据：
- **研发设计数据**：BOM清单、关键物料规格与供应商、图纸尺寸与公差、ECN变更记录。
- **工艺与试产数据**：各工序核心参数（温度/压力/速度/时间等）、设备编号与维护状态、环境数据（温湿度）、试产调试记录。
- **真实结果数据**：历史各批次的实际一次合格率（FPY）、主要不良类型与数量。
⚠️ 关键点：数据必须能跨系统（PLM/MES/QMS）对齐；样本量最好有几百至上千条历史批次记录，否则模型“学不到规律”。

**2. 适合先做什么最小版本（MVP）**
不要一上来做全厂通用平台，按“单点突破→验证→复制”走：
- **选1个试点**：挑历史数据最全、工艺最成熟的1款产品或1条产线。
- **做最简功能**：输入当前设定的关键参数 → 输出 `预测合格率 ± 置信区间` + `拉低合格率的Top 3风险参数`。
- **技术策略**：先用轻量树模型（如XGBoost/LightGBM）快速跑通，不碰深度学习或复杂大模型。
- **验收标准**：与真实试产结果对比，预测误差控制在业务可接受范围（如±2%~3%），且风险提示符合工艺常识，即算跑通。

**3. 应该先找谁参与**
落地成败不在算法多高级，而在“业务懂数据+数据能落地”。先拉一个**核心三人组**：
- **工艺/质量工程师**（业务大脑）：负责挑关键参数、解释数据含义、验证模型结果是否符合物理规律。没有他们，模型极易学到“伪相关”。
- **数据工程师/IT开发**（数据管道）：负责从PLM/MES/QMS等系统抽数、清洗、对齐字段，解决“数据孤岛”和脏数据问题。
- **算法工程师/数据科学家**（模型构建）：负责特征工程、训练评估、输出可解释的预测结果。
📌 必须有一位**业务负责人（如质量总监或研发经理）**全程参与，明确“误差多少算合格”“预测结果如何介入量产决策”，否则系统很容易变成“实验室玩具”。


In [20]:
# 将以上步骤封装成一个函数，输入需求，输出判断结果
def 需求判断助手(需求):
    step1 = ask_ai(f"{需求} 的本质是什么")
    
    step2 = ask_ai(f"""
    {step1}
    请判断这是AI项目还是IT项目，并说明原因
    """)
    
    step3 = ask_ai(f"""
    {step1}
    {step2}
    请给出一个简单的落地建议
    """)
    
    return step3


In [ ]:
# 调用需求判断助手函数，输入一个需求，输出判断结果
result = 需求判断助手("希望系统自动识别异常工艺参数")
print(result)